In [46]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import euclidean
from scipy.stats import linregress
from itertools import combinations 

base_dir        = '/Users/lilymarino/Documents/GitHub/t-c_continuum'
data_filename   = 'dataset-ariWave1_data-allPhysio_detail-standardizedChangeScores.csv'
ref_filename    = 'dataset-arb_data-allPhysio_detail-2clusterStandardizedChangeScoreMeans.csv'
df              = pd.read_csv(os.path.join(base_dir, 'data', data_filename))
ref_means       = pd.read_csv(os.path.join(base_dir, 'ref_data', ref_filename))
plot_by_combo   = True # generates 1 png per combo with subplots for each subject
plot_by_sub     = True # generates 1 png per subject with subplots for each combo

dataset     = data_filename.split('dataset-')[1].split('_')[0] # extract name of test dataset
ref_dataset = ref_filename.split('dataset-')[1].split('_')[0]  # extract name of reference dataset

In [41]:
# all available metrics:
        # SCR       # SCL       # IBI       # RSA       # RR 
        # PEP       # LVET      # CO        # MAP       # TPR

# generate all unique combinations utilizing 2-8 metrics
# exclusions: 
    # RR   - confound do to speech during TSST
    # LVET - not seemingly very relevant?
all_relevant_metrics = ['SCR_delta', 'SCL_delta', 'IBI_delta', 'RSA_delta', 'MAP_delta', 'CO_delta', 'PEP_delta', 'TPR_delta']
all_metric_combos    = []
for n in range(2, 9):
    combos = list(combinations(all_relevant_metrics, n))
    all_metric_combos.extend(combos)

In [ ]:
all_results = []

for combo_idx, metrics in enumerate(all_metric_combos):

    # generate combo name
    combo_name = f"Combo {combo_idx + 1}/{len(all_metric_combos)}: {list(metrics)}"
    metrics    = list(metrics)
    print(f"Processing Combo {combo_idx + 1}/{len(all_metric_combos)}")

    results_combo = []

    # calculate euclidean distances per subject per trial
    for index, row in df.iterrows():

        subject = row['subject']
        trial   = row['trial']

        delta_values         = row[metrics].values
        trial_means          = ref_means[ref_means['trial'] == trial]
        feature_names        = [f.replace('_delta', '') for f in metrics]
        trial_means_filtered = trial_means[trial_means['metric'].isin(feature_names)]

        challenge_means = trial_means_filtered['challenge_cluster_means'].values
        threat_means    = trial_means_filtered['threat_cluster_means'].values

        # calculate euclidean distance between subject per trial and reference per trial mean change scores
        challenge_dist = euclidean(delta_values, challenge_means)
        threat_dist    = euclidean(delta_values, threat_means)

        results_combo.append({
            'combo_idx':  combo_idx,
            'combo_name': combo_name,
            'subject':    subject,
            'trial':      trial,
            'challenge_dist': challenge_dist,
            'threat_dist':    threat_dist
        })

    # calculate similarity scores (1 / distance scores)
    results_combo = pd.DataFrame(results_combo)
    results_combo['challenge_sim']      = 1 / results_combo['challenge_dist']
    results_combo['threat_sim']         = 1 / results_combo['threat_dist']
    results_combo['challenge_sim_norm'] = results_combo['challenge_sim'] / (results_combo['challenge_sim'] + results_combo['threat_sim'])
    results_combo['threat_sim_norm']    = results_combo['threat_sim']    / (results_combo['challenge_sim'] + results_combo['threat_sim'])

    # calculate slope of challenge similarity
    slope_df = (
        results_combo.groupby('subject')
        .apply(lambda g: linregress(g['trial'], g['challenge_sim_norm']).slope)
        .reset_index()
        .rename(columns={0: 'challenge_sim_slope'})
        )
    
    results_combo = results_combo.merge(slope_df, on='subject', how='left')

    all_results.append(results_combo)

all_results = pd.concat(all_results, ignore_index=True)

Processing Combo 1/247
Processing Combo 2/247
Processing Combo 3/247
Processing Combo 4/247
Processing Combo 5/247
Processing Combo 6/247
Processing Combo 7/247
Processing Combo 8/247
Processing Combo 9/247
Processing Combo 10/247
Processing Combo 11/247
Processing Combo 12/247
Processing Combo 13/247
Processing Combo 14/247
Processing Combo 15/247
Processing Combo 16/247
Processing Combo 17/247
Processing Combo 18/247
Processing Combo 19/247
Processing Combo 20/247
Processing Combo 21/247
Processing Combo 22/247
Processing Combo 23/247
Processing Combo 24/247
Processing Combo 25/247
Processing Combo 26/247
Processing Combo 27/247
Processing Combo 28/247
Processing Combo 29/247
Processing Combo 30/247
Processing Combo 31/247
Processing Combo 32/247
Processing Combo 33/247
Processing Combo 34/247
Processing Combo 35/247
Processing Combo 36/247
Processing Combo 37/247
Processing Combo 38/247
Processing Combo 39/247
Processing Combo 40/247
Processing Combo 41/247
Processing Combo 42/247
P

In [48]:
all_results

,combo_idx,combo_name,subject,trial,challenge_dist,threat_dist,challenge_sim,threat_sim,challenge_sim_norm,threat_sim_norm,challenge_sim_slope
0,0,"Combo 1/247: ['SCR_delta', 'SCL_delta']",sub-003,1,1.538385,1.393217,0.650032,0.717763,0.475241,0.524759,0.004075
1,0,"Combo 1/247: ['SCR_delta', 'SCL_delta']",sub-003,2,0.730110,0.609281,1.369656,1.641278,0.454894,0.545106,0.004075
2,0,"Combo 1/247: ['SCR_delta', 'SCL_delta']",sub-003,3,0.983933,0.909720,1.016330,1.099239,0.480405,0.519595,0.004075
3,0,"Combo 1/247: ['SCR_delta', 'SCL_delta']",sub-003,4,0.680262,0.628739,1.470021,1.590485,0.480320,0.519680,0.004075
4,0,"Combo 1/247: ['SCR_delta', 'SCL_delta']",sub-006,1,0.932514,1.077023,1.072369,0.928485,0.535956,0.464044,-0.022631
...,...,...,...,...,...,...,...,...,...,...,...
45443,246,"Combo 247/247: ['SCR_delta', 'SCL_delta', 'IBI...",sub-067,4,1.889156,2.084153,0.529337,0.479811,0.524538,0.475462,-0.008815
45444,246,"Combo 247/247: ['SCR_delta', 'SCL_delta', 'IBI...",sub-068,1,2.727999,2.630206,0.366569,0.380198,0.490874,0.509126,-0.005211
45445,246,"Combo 247/247: ['SCR_delta', 'SCL_delta', 'IBI...",sub-068,2,1.947805,2.428725,0.513398,0.411739,0.554943,0.445057,-0.005211
45446,246,"Combo 247/247: ['SCR_delta', 'SCL_delta', 'IBI...",sub-068,3,3.229093,2.718975,0.309685,0.367786,0.457119,0.542881,-0.005211


In [ ]:
if plot_by_combo:

    plot_by_combo_dir = os.path.join(base_dir, 'results', 'plots_by_combo')
    os.makedirs(plot_by_combo_dir, exist_ok=True)

    for combo_idx, combo_df in all_results.groupby('combo_idx'):

        combo_name = combo_df['combo_name'].iloc[0]
        subjects   = combo_df['subject'].unique()
        n_subjects = len(subjects)
        n_cols     = int(np.ceil(np.sqrt(n_subjects)))
        n_rows     = int(np.ceil(n_subjects / n_cols))

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 2.5))
        axes = axes.flatten()
        fig.suptitle(f'Cluster Similarity\n {combo_name}\n Dataset: {dataset}\n Reference Dataset: {ref_dataset}', fontsize = 10, y = 0.995)

        for idx, subject in enumerate(subjects):
            
            sub_data = combo_df[combo_df['subject'] == subject]
            axes[idx].plot(sub_data['trial'], sub_data['challenge_sim_norm'], marker='o', color='#482173', linewidth=1.5)
            axes[idx].plot(sub_data['trial'], sub_data['threat_sim_norm'],    marker='s', color='#29af7f', linewidth=1.5)
            axes[idx].set_title(subject, fontsize=9)
            axes[idx].set_xticks([1, 2, 3, 4])
            axes[idx].set_ylim([0, 1])
            axes[idx].grid(True, alpha=0.3)

            if idx >= len(subjects) - n_cols:
                axes[idx].set_xlabel('Trial', fontsize=8)
            if idx % n_cols == 0:
                axes[idx].set_ylabel('Cluster Similarity', fontsize=8)

        for idx in range(n_subjects, len(axes)):
            axes[idx].axis('off')

        fig.legend(['Challenge', 'Threat'], loc='upper right', bbox_to_anchor=(0.98, 0.98), fontsize=10)
        plt.tight_layout()
        plt.savefig(os.path.join(plot_by_combo_dir, f'dataset-{dataset}_data-combo{combo_idx + 1}_detail-plotByCombo.png'), dpi=150, bbox_inches='tight')
        plt.close()

In [ ]:
if plot_by_sub:
    
    plot_by_sub_dir = os.path.join(base_dir, 'results', 'plots_by_sub')
    os.makedirs(plot_by_sub_dir, exist_ok=True)

    for subject, sub_df in all_results.groupby('subject'):

        combos   = sub_df['combo_idx'].unique()
        n_combos = len(combos)
        n_cols   = int(np.ceil(np.sqrt(n_combos)))
        n_rows   = int(np.ceil(n_combos / n_cols))

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 2.5))
        axes = axes.flatten()
        fig.suptitle(f'Cluster Similarity\n Subject: {subject}\n Dataset: {dataset}\n Reference Dataset: {ref_dataset}', fontsize = 10, y = 0.995)

        for idx, combo_idx in enumerate(combos):
            
            combo_data = sub_df[sub_df['combo_idx'] == combo_idx]
            combo_name = combo_data['combo_name'].iloc[0]
            axes[idx].plot(combo_data['trial'], combo_data['challenge_sim_norm'], marker='o', color='#482173', linewidth=1.5)
            axes[idx].plot(combo_data['trial'], combo_data['threat_sim_norm'],    marker='s', color='#29af7f', linewidth=1.5)
            axes[idx].set_title(f'Combo {combo_idx + 1}', fontsize=9)
            axes[idx].set_xticks([1, 2, 3, 4])
            axes[idx].set_ylim([0, 1])
            axes[idx].grid(True, alpha=0.3)

            if idx >= n_combos - n_cols:
                axes[idx].set_xlabel('Trial', fontsize=8)
            if idx % n_cols == 0:
                axes[idx].set_ylabel('Cluster Similarity', fontsize=8)

        for idx in range(n_combos, len(axes)):
            axes[idx].axis('off')

        fig.legend(['Challenge', 'Threat'], loc='upper right', bbox_to_anchor=(0.98, 0.98), fontsize=10)
        plt.tight_layout()
        plt.savefig(os.path.join(plot_by_sub_dir, f'dataset-{dataset}_data-{subject}_detail-plotBySub.png'), dpi=150, bbox_inches='tight')
        plt.close()

plot_by_sub complete.
